# Mandatory Assignment 1: Training and Evaluating Baseline Models
#### Group: Prakmas

In [ ]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import weighted, average
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV


## Description of the dataset
The dataset that the task is based on is the "Student Performance Factors".
"The dataset provides a comprehensive overview of various factors affecting student performance in exams.
It includes information on study habits, attendance, parental involvement, and other aspects influencing academic success." - https://www.kaggle.com/datasets/lainguyn123/student-performance-factors

The target value is the exam score. To create a classification problem out of the dataset, this value will be encoded into categorical data.

The dataset does not have a real source. It instead claims to be created for simulating realistic scenarios for analyzing student performance factors. As such, the group has designed a fictional scenario to base this task on. The scenario is described in the next paragraph. To make the dataset more balanced, and since the data is generated rather than real, some adjustments has been made to the grade scale. The grades are created in a non-linear way from the exam score to create a more realistic grade distribution.

### Scenario
Exams done in the first half of the year can end up having a short time between grades being finalized and admission to schools and universities. To combat this problem the Norwegian school system has developed a prediction model for how well a student will do based on a set of data about the student. The grade given by the model is then used by the government when assigning students to different schools.

For this scenario it is considered more important for the model to get a higher grade correct than giving a higher grade than the real value. This will keep more opportunities available to students before the real grades are finalized. There is then more closure after the final grades are released, and more ambiguity when just the predicted grades available. This may differ depending on how this system is used. If it is used in a way that when too many students are predicted to get the same grade, the course will be temporarily be overfilled until the final grades are available, then it would work like intended. This is the basis for how the models are evaluated.

In [ ]:
# Read the dataset
student_dataset = pd.read_csv('Dataset/StudentPerformanceFactors.csv')
# Uses existing column names baked into csv file.

# Show the first 5 rows to validate that the data import has been successful
student_dataset.head()

In [ ]:
# Look at the implementation details, data types and missing data
student_dataset.info()

## Data Processing

### Missing Values


In [ ]:
# Count the number of missing values in the dataset
student_dataset.isna().sum()

In [ ]:
# Check how many rows include at least one missing value.
student_dataset.isna().any(axis=1).sum()

In [ ]:
# Remove rows with missing values from the student performance dataset.
student_dataset = student_dataset.dropna()

# Check to see if there still are features with missing values.
student_dataset.isna().sum()

### Separating the dataset into features and targets

In [ ]:
features = student_dataset.drop('Exam_Score', axis=1)
targets = student_dataset['Exam_Score']
targets = targets.to_frame()
print(targets)

### Split dataset
Using the train_test_split() function from sklearn twice to first split the data into train/test sets.
Then it is further split a second time into train/test/validation sets by sending in the train set.
NOT STRATIFIED!


In [ ]:
features_train, features_test, targets_train, targets_test = train_test_split(
    features, targets,
    train_size=0.8, # Splitting 80% of the dataset into training. 0.8 * 1.0 = 0.8
    test_size=0.2, # Splitting 20% of the dataset into testing. 0.2 * 1.0 = 0.2
    random_state=21
)

In [ ]:
features_train, features_validation, targets_train, targets_validation = train_test_split(
    features_train, targets_train,
    train_size=0.75, # Splitting the 80% training set to only 60%. 0.75 * 0.80 = 0.60. One ends up with 60% training data.
    test_size=0.25, # Splitting the 80% training set to 20% validation data. 0.25 * 0.80 = 0.20. One ends up with 20% validation data.
    random_state=21
)

To summarize, The results from splitting the data are as following:
- 60% training
- 20% test data
- 20% validation data

To summarize, The results from splitting the data are as following:
- 60% training
- 20% test data
- 20% validation data

In [ ]:
print(features_train.columns.tolist())

### Encoding
The data is encoded in two main ways. The first one is using an ordinal encoder. This is used for the data that a order which should be retained for the ML-model to take into account. These values are specified in the "categories" parameter of the method. This specification allows the data to be contextually encoded so that the higher values is greater than the lower values. The other way of encoding which is used is a form of binary encoding. This is used for the data that only has two options, such as 'yes' or 'no' and 'female' or 'male'. These features uses a more direct approach of encoding, using pandas' replace function on the datasets.

The encoding is done on split datasets. The encoding will be done in the same way on all three datasets, but doing it after minimizes the risk of data leakage. Additionally, the original dataframe is kept intact, leaving only data in the split dataframes replaced. Originally the encoded data was added to a seperate column to retain the categorical data. Since the categorical data is retained in the original dataframe, it was deemed to be too much work for little gain to add an extra column. The encoded values will now simply replace the old value in the same column.

In [ ]:
# - - - Ordinal data - - -
# - - Parental Involvement - -
encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
features_train['Parental_Involvement']= encoder.fit_transform(features_train[['Parental_Involvement']])
features_validation['Parental_Involvement']= encoder.fit_transform(features_validation[['Parental_Involvement']])
features_test['Parental_Involvement']= encoder.fit_transform(features_test[['Parental_Involvement']])


# Previous way of encoding (straight into the unsplit dataframe)
#encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
#student_dataset['Parental_Involvement_Encoded'] = encoder.fit_transform(student_dataset[['Parental_Involvement']])
#print(student_dataset[['Parental_Involvement','Parental_Involvement_Encoded']])

In [ ]:
# - - Access to Resources - -
encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
features_train['Access_to_Resources']= encoder.fit_transform(features_train[['Access_to_Resources']])
features_validation['Access_to_Resources']= encoder.fit_transform(features_validation[['Access_to_Resources']])
features_test['Access_to_Resources']= encoder.fit_transform(features_test[['Access_to_Resources']])

In [ ]:
# - - Motivation_Level - -
encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
features_train['Motivation_Level']= encoder.fit_transform(features_train[['Motivation_Level']])
features_validation['Motivation_Level']= encoder.fit_transform(features_validation[['Motivation_Level']])
features_test['Motivation_Level']= encoder.fit_transform(features_test[['Motivation_Level']])

In [ ]:
# - - Family_Income - -
encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
features_train['Family_Income']= encoder.fit_transform(features_train[['Family_Income']])
features_validation['Family_Income']= encoder.fit_transform(features_validation[['Family_Income']])
features_test['Family_Income']= encoder.fit_transform(features_test[['Family_Income']])

In [ ]:
# - - Teacher_Quality - -
encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
features_train['Teacher_Quality']= encoder.fit_transform(features_train[['Teacher_Quality']])
features_validation['Teacher_Quality']= encoder.fit_transform(features_validation[['Teacher_Quality']])
features_test['Teacher_Quality']= encoder.fit_transform(features_test[['Teacher_Quality']])
# Since we removed the missing values beforehand, the encoding goes smoothly without interruptions

In [ ]:
# - - Peer_Influence - -  
encoder = OrdinalEncoder(categories=[['Negative', 'Neutral', 'Positive']])
features_train['Peer_Influence']= encoder.fit_transform(features_train[['Peer_Influence']])
features_validation['Peer_Influence']= encoder.fit_transform(features_validation[['Peer_Influence']])
features_test['Peer_Influence']= encoder.fit_transform(features_test[['Peer_Influence']])

In [ ]:
# - - Parental_Education_Level - -
encoder = OrdinalEncoder(categories=[['High School', 'College', 'Postgraduate']])
features_train['Parental_Education_Level']= encoder.fit_transform(features_train[['Parental_Education_Level']])
features_validation['Parental_Education_Level']= encoder.fit_transform(features_validation[['Parental_Education_Level']])
features_test['Parental_Education_Level']= encoder.fit_transform(features_test[['Parental_Education_Level']])

In [ ]:
# - - Distance_from_Home - -
encoder = OrdinalEncoder(categories=[['Near', 'Moderate', 'Far']])
features_train['Distance_from_Home']= encoder.fit_transform(features_train[['Distance_from_Home']])
features_validation['Distance_from_Home']= encoder.fit_transform(features_validation[['Distance_from_Home']])
features_test['Distance_from_Home']= encoder.fit_transform(features_test[['Distance_from_Home']])

In [ ]:
# - - - Binary data - - -
# - - Gender - -
print("A quick check to see if the encoding worked: ")
print("Original data:")
print(features_train.Gender.value_counts(), "\n")
features_train['Gender'] = features_train['Gender'].replace({'Female':0, 'Male':1})
print("Encoded data:")
print(features_train.Gender.value_counts())
features_validation['Gender']= features_validation['Gender'].replace({'Female':0, 'Male':1})
features_test['Gender']= features_test['Gender'].replace({'Female':0, 'Male':1})

In [ ]:
# - - Extra Curricular Activities - -

features_train['Extracurricular_Activities'] = features_train['Extracurricular_Activities'].replace({'No':0, 'Yes':1}).astype(int)
features_validation['Extracurricular_Activities'] = features_validation['Extracurricular_Activities'].replace({'No':0, 'Yes':1}).astype(int)
features_test['Extracurricular_Activities'] = features_test['Extracurricular_Activities'].replace({'No':0, 'Yes':1}).astype(int)


In [ ]:
# - - Learning_Disabilites - -
features_train['Learning_Disabilities'] = features_train['Learning_Disabilities'].replace({'No':0, 'Yes':1}).astype(int)
features_validation['Learning_Disabilities'] = features_validation['Learning_Disabilities'].replace({'No':0, 'Yes':1}).astype(int)
features_test['Learning_Disabilities'] = features_test['Learning_Disabilities'].replace({'No':0, 'Yes':1}).astype(int)


In [ ]:
# - - School_Type - -
features_train['School_Type'] = features_train['School_Type'].replace({'Public':0, 'Private':1}).astype(int)
features_validation['School_Type'] = features_validation['School_Type'].replace({'Public':0, 'Private':1}).astype(int)
features_test['School_Type'] = features_test['School_Type'].replace({'Public':0, 'Private':1}).astype(int)


In [ ]:
# - - Internet_Access - -
features_train['Internet_Access'] = features_train['Internet_Access'].replace({'No':0, 'Yes':1}).astype(int)
features_validation['Internet_Access'] = features_validation['Internet_Access'].replace({'No':0, 'Yes':1}).astype(int)
features_test['Internet_Access'] = features_test['Internet_Access'].replace({'No':0, 'Yes':1}).astype(int)


## Analysis
### Univariate Analysis

In [ ]:
# Creating an encoded dataset to use for analysis, ignores index to avoid duplicate indices
analysis_dataset = pd.concat([features_train, features_validation, features_test], ignore_index=True)

print(analysis_dataset['Internet_Access'])

In [ ]:
#Plotting the columns with numeric values into histograms
fig, ax = plt.subplots(1,1, figsize=(10,25))
analysis_dataset.hist(ax=ax)

We can clearly see that the previous exam scores were much more evenly spread, while the final exam scores were vastly in the range of 60-80 points. The exam scores seem to follow a normal distribution, while the previous scores are more of a uniform distribution. 

In [ ]:
#Plotting a KDE of Previous Scores and Final Score to get a closer look at the distribution
analysis_dataset['Previous_Scores'].plot.kde()
student_dataset['Exam_Score'].plot.kde()

It's clear that the exam scores were much more concentrated around 60-80 points than the previous scores. As of now it seems like the exam was much more difficult than previous tests. We originally thought the exam might be curved, but that doesn't make much sense because these are the actual scores, not the grades. 

In [ ]:
#Plotting the boxplots for the dataset to see which ones are close in the distribution, then separating them into plots with fewer boxplots. We see that Attendance, Previous_Scores and Exam_Scores seem close, and Hours_Studied matches well with Sleep_Hours.
fig, ax = plt.subplots(1,1,figsize=(30, 30))
analysis_dataset.boxplot()

In [ ]:
Score_features = ["Attendance","Previous_Scores"]
analysis_dataset[Score_features].boxplot()

The Exam_Score box plot is much more narrow than the other two, and it's the only one with any outliers. There are people that have scored 75+ on the exam, but there are not that many of them. As there are no exam scores below around 54, it seems like the test might have had some easy questions that most students were able to answer, and some very hard questions that pushed a few students into the outliers. As for the attendance, it seems like there was a minimum of 60% mandatory attendance needed to take the exam. 

In [ ]:
Score_features = ["Hours_Studied","Sleep_Hours"]
analysis_dataset[Score_features].boxplot()

Most students study about 15-25 hours each week, and sleep around 7-8 hours each day. There are some outliers who studied above 37 hours each week and below 4 hours each week. 

In [ ]:

#Plotting the rest of the features together as they seem similar in how they're distributed. 
Rest_features = [
    "Tutoring_Sessions",
    "Physical_Activity",
    "Parental_Involvement",
    "Access_to_Resources",
    "Extracurricular_Activities",
    "Motivation_Level",
    "Internet_Access",
    "Family_Income"
]
fig, ax = plt.subplots(1,1,figsize=(30, 30))
analysis_dataset[Rest_features].boxplot()

When it comes to tutoring sessions, it seems like the majority of students had between 0-3 sessions, with outliers reaching all the way up to 8 tutoring sessions. As for Internet Access, there are very few students with no internet access. The Internet_Access_Encoded box is barely visible so let's put it by itself to see clearly what it says. 

In [ ]:
analysis_dataset[['Internet_Access']].boxplot()

### Multivariate Analysis
In this part of the analysis, the features will be analyzed in relation to each other.
#### Correlation



In [ ]:
# Correlation gradient map.

# Computes the pairwise correlation between each feature.
# This method will only use numerical features.
# correlation = student_dataset.drop('Exam_Score', axis=1).corr(numeric_only=True)
correlation = analysis_dataset.corr(numeric_only=True)
# Shows the correlation between the features in a heatmap.
plt.figure(figsize=(20, 20))
sns.heatmap(correlation, cmap='coolwarm', annot=True, fmt='.3f', vmin=-1, vmax=1)


#### Results
The results of the correlation graph shows that none of the features are correlated in any notable degree. This means that all the individual features provides unique data to the model, rather than a two or more features giving the same information and making them redundant.

### Scatterplot
To further investigate the correlation, can scatterplots be used. These will show the correlation between two features. The more linear the scatterplot is, the more correlated the features are. Since there is virtually no correlation between the features of the dataset, the scatterplots that will be shown are have many scattered dots. This is an indication of low correlation between features.

To showcase some scatterplots, pairs of features that might be thought to have correlation, if taking the domain into account, is shown.


In [ ]:
# Scatterplot.
fig, ax = plt.subplots(1, 3, figsize=(20,5))

ax[0].scatter(analysis_dataset['School_Type'], analysis_dataset['Family_Income'])
ax[1].scatter(analysis_dataset['Previous_Scores'], analysis_dataset['Attendance'])
ax[2].scatter(analysis_dataset['Previous_Scores'], analysis_dataset['Hours_Studied'])


In [ ]:
# Scatterplot test
plt.scatter(analysis_dataset['School_Type'], analysis_dataset['Family_Income'])

In [ ]:
# Encode the categorical target using one-hot encoding
# This would be the preferred way to encode (or dummy encoding) targets for 
# ML in general. Scikit learn does not require this. In fact, it can cause issues
# In scikit-learn, you should use LabelEncoder instead
# All categorical features must be encoded in general.
targets_onehot = pd.get_dummies(targets)
targets_onehot

In [ ]:
# Creates a new subset of the dataframe which excludes the non-encoded categorial data.
# This is needed for the baseline model inputs.
student_numerical_dataset = student_dataset.select_dtypes(exclude="string")

# Check to see if the new dataset includes  as intended.
student_numerical_dataset.head()

## Data Processing part 2
### Normalization
The inputs are normalized using mean normalization. Inputs that are encoded as categories are not normalized. This is done after the analysis to retain the original values from the dataset in the analysis.


In [ ]:
# Inputs that are to be normalized. These will be passed into the different features dataFrames.
inputs_to_normalize = [
    'Hours_Studied',
    'Attendance',
    'Sleep_Hours',
    'Previous_Scores',
    'Tutoring_Sessions',
    'Physical_Activity'
]

In [ ]:
# Finding the mean, minimum, and maximum in the features_train dataset.
train_mean = features_train[inputs_to_normalize].mean()
train_min = features_train[inputs_to_normalize].min()
train_max = features_train[inputs_to_normalize].max()

In [ ]:
# Mean normalization formula
def mean_normalization_func(data_frame: pd.DataFrame) -> pd.DataFrame:
    normalized_dataframe = (data_frame - train_mean) / (train_max - train_min)
    return normalized_dataframe


In [ ]:
# Normalizing the feature which are split by using the function above.
features_train[inputs_to_normalize] = (mean_normalization_func(features_train[inputs_to_normalize]))

features_validation[inputs_to_normalize] = (mean_normalization_func(features_validation[inputs_to_normalize]))

features_test[inputs_to_normalize] = (mean_normalization_func(features_test[inputs_to_normalize]))

In [ ]:
# Printing to check if normalization has been done correctly
print(features_train[inputs_to_normalize].describe())

In [ ]:
# Printing the first few values to check if normalization has been done correctly
print(features_train[inputs_to_normalize].head())

From both:<br>
   - print(features_train[inputs_to_normalize].describe())
   - print(features_train[inputs_to_normalize].head())

one can see that the normalization has been done correctly,
as the values in the "mean" row represent numbers close to zero.
The reason why they are not zero is because of floating-point precision as we had to do mathematical operations on a computer to perform the mean normalization.
<br>We can see that the min max are also correct by taking a colum and subtracting max and min.
Attendance = 0.4993662 - (-0.5006338) = 1

We can also see from the .head() that the normalization has been successful as the values are strictly between 1 and -1.

From both:<br>
   - print(features_train[inputs_to_normalize].describe())
   - print(features_train[inputs_to_normalize].head())

one can see that the normalization has been done correctly,
as the values in the "mean" row represent numbers close to zero.
The reason why they are not zero is because of floating-point precision as we had to do mathematical operations on a computer to perform the mean normalization.
<br>We can see that the min max are also correct by taking a colum and subtracting max and min.
Attendance = 0.4993662 - (-0.5006338) = 1

We can also see from the .head() that the normalization has been successful as the values are strictly between 1 and -1.

### Target value
#### Exam score
The exam score will first be translated into a category to make the task connected to the dataset into a classification problem.
This will be done by assigning grades to the exam scores. As can be seen from the Exam_Score histogram, the grades were very concentrated around the 60s. A non-linear grading scale has therefore been used to compensate for this. This will make the dataset more balanced similarly to how grading on a curve is done. 
Afterwards, the grades will be encoded using ordered encoding to preserve the grade order.

In [ ]:
def gradesetter(points):
    if points < 60:
        return 'Fail'
    elif 60 <= points < 65:
        return 'D'
    elif 65 <= points < 69:
        return 'C'
    elif 69 <= points < 74:
        return 'B'
    else:
        return 'A'


targets_train['Exam_Score'] = targets_train['Exam_Score'].map(gradesetter)
targets_validation['Exam_Score'] = targets_validation['Exam_Score'].map(gradesetter)
targets_test['Exam_Score'] = targets_test['Exam_Score'].map(gradesetter)

In [ ]:
# Ordinal Encoding of the targets
encoder = OrdinalEncoder(categories=[['Fail', 'D', 'C', 'B', 'A']])
targets_train['Exam_Score']= (encoder.fit_transform(targets_train[['Exam_Score']]).astype(int))
targets_validation['Exam_Score']= (encoder.transform(targets_validation[['Exam_Score']]).astype(int))
targets_test['Exam_Score']= (encoder.transform(targets_test[['Exam_Score']]).astype(int))

In [ ]:
targets_train.hist()

## Baseline Models
This is the part where the three baseline models will be trained and tuned.
The baseline models include decision trees, naive bayes and linear regression.

### Decision Trees

In [ ]:
# Train a decision tree with default hyperparameters
dt_default = DecisionTreeClassifier(random_state=21)
dt_default.fit(features_train, targets_train['Exam_Score'])


The outputs from over results in what we call overfitting. Overfitting means the model has essentially memorized the training data instead of learning general patterns that hold up on new, unseen data. It's easy to spot when training data, and validation has a big gap, like the results we just got.

In [ ]:
# Tune max_depth hyperparameter
# Starting with max_depth=5
dt_maxdepth_5 = DecisionTreeClassifier(max_depth=5, random_state=21)
dt_maxdepth_5.fit(features_train, targets_train['Exam_Score'])


However, when we limit model to a max_depth, forces the tree to stop after N splits and make its prediction based on a broader, less specific group of training examples at that leaf.

In [ ]:
# Trying with 3,5,7,10 and None for more depths/criterias
for depth in [3, 5, 7, 10, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=21)
    dt.fit(features_train, targets_train['Exam_Score'])
    print(f'max_depth={depth}') #f updates the string with the value of depth


In [ ]:
#A more cleaner way of presenting the results. 
#Intorducing max_depth=15 to see if it improves the model
results = []
for depth in [3, 5, 7, 10, 15, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=21)
    dt.fit(features_train, targets_train['Exam_Score'])
    train_acc = accuracy_score(targets_train['Exam_Score'], dt.predict(features_train))
    val_acc = accuracy_score(targets_validation['Exam_Score'], dt.predict(features_validation))
    results.append({'max_depth': depth, 'train_acc': train_acc, 'val_acc': val_acc})

pd.DataFrame(results)

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_leaf': [1, 5, 10, 20, 30, 50],
    'min_samples_split': [2, 10, 20], #2 is default by sklearn
}
#Total combinations: 6×4×3×2= 144 different decision trees:
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=21),
    param_grid,
    scoring='f1_weighted',     
    n_jobs=-1                  #forces to use all CPU cores, which speeds this up a lot (144 trees takes time)
)
grid_search.fit(features_train, targets_train['Exam_Score'])

print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)
dt_best = grid_search.best_estimator_


### Naive Bayes

### Linear Regression

In [ ]:
#Vi har et multiclass problem, bruker derfor SGDClassifier
sgd_default = SGDClassifier()
sgd_default.fit(features_train, targets_train['Exam_Score'])

In [ ]:
# Tuning a SGDClassifier model
sgd_tuned = SGDClassifier(alpha=1e-4,penalty = "l1", random_state=21)
sgd_tuned.fit(features_train, targets_train['Exam_Score'])

## Evaluation and Comparison
For this part the models will be compared and evaluated on different metrics.

We need to decide which metric fits our scenario.

In [ ]:
# Tentative functions for tuning models.
# Used to evaluate hyperparameter tuning
def validation(model):
    train_predictions = model.predict(features_train)
    validation_predictions = model.predict(features_validation)
    print(f'Training Accuracy: {accuracy_score(targets_train["Exam_Score"], train_predictions,):.2f} Validation: {accuracy_score(targets_validation["Exam_Score"], validation_predictions):.2f}')
    print(f'Training Precision: {precision_score(targets_train["Exam_Score"], train_predictions, average="weighted", zero_division=0):.2f} Validation: {precision_score(targets_validation["Exam_Score"], validation_predictions, average="weighted", zero_division=0):.2f}')
    print(f'Training Recall: {recall_score(targets_train["Exam_Score"], train_predictions, average="weighted", zero_division=0):.2f} Validation: {recall_score(targets_validation["Exam_Score"], validation_predictions, average="weighted", zero_division=0):.2f}')
    print(f'Training F1: {f1_score(targets_train["Exam_Score"], train_predictions, average="weighted", zero_division=0):.2f} Validation: {f1_score(targets_validation["Exam_Score"], validation_predictions, average="weighted", zero_division=0):.2f}')

In [ ]:
# Looking at how the tuned sgd model performs in training
validation(sgd_tuned)

### Decision Trees

In [ ]:
# Perform validation
validation(dt_default)

validation(dt_maxdepth_5)

validation(dt)

validation(dt_best)